In [ ]:
import os
import hashlib
import pickle
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings

from kaggle_modules.i import input_dir, model_dir, output_dir

from sklearn.base import clone, BaseEstimator, TransformerMixin
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

train = pd.read_csv(input_dir + "train.csv", index_col="id")
test = pd.read_csv(input_dir + "test.csv", index_col="id")
origin = pd.read_csv(input_dir + "diabetes_dataset.csv")

for col in train.select_dtypes(include="object").columns:
    train[col] = train[col].astype("category")
    test[col] = test[col].astype("category")
for col in origin.select_dtypes(include="object").columns:
    origin[col] = origin[col].astype("category")

Target_Col = "diagnosed_diabetes"

X = train.iloc[:, :-1]
y = train[Target_Col]
X_test = test

X_orig = origin[X.columns]
y_orig = origin[Target_Col]

In [ ]:
plt.figure(figsize=(10,8))
sns.countplot(data=train, x=train.iloc[:, -1])
plt.show()

In [ ]:
df_processed = pd.DataFrame(data=OrdinalEncoder().fit_transform(train), index=train.index, columns=train.columns)
origin_processed = pd.DataFrame(data=OrdinalEncoder().fit_transform(origin), index=origin.index, columns=origin.columns)

plt.figure(figsize=(15,8))
sns.heatmap(data=df_processed.corr(),fmt=".2f", annot=True, cmap="icefire")
plt.show()
plt.figure(figsize=(18,13))
sns.heatmap(data=origin_processed.corr(),fmt=".2f", annot=True, cmap="icefire")
plt.show()

In [ ]:
class Features(BaseEstimator, TransformerMixin):
    def __init__(self):
        self

    def fit(self, X, y=None):
        X = X.copy()
        self.exerciseMean = X["physical_activity_minutes_per_week"].mean()
        return self

    def transform(self, X):
        X = X.copy()

        X["exerciseRatio"] = X["physical_activity_minutes_per_week"] / self.exerciseMean
        return X

In [ ]:
xgb_base = Pipeline(
    [
        ("features", Features()),
        (
            "model",
            XGBClassifier(
                random_state=42,
                booster="dart",
                n_estimators=1000,
                learning_rate=0.03,
                max_depth=7,
                subsample=0.8,
                colsample_bytree=0.8,
                enable_categorical=True,
                device="gpu",
                n_jobs=7,
            ),
        ),
    ]
)

lgb_params = {
    "boosting_type": "dart",
    "learning_rate": 0.16722132581464857,
    "n_estimators": 2896,
    "num_leaves": 366,
    "max_depth": 12,
    "min_data_in_leaf": 129,
    "min_child_weight": 0.2664914177258311,
    "min_split_gain": 4.263018771802797,
    "lambda_l1": 5.676843225483512e-06,
    "lambda_l2": 0.0005589915238473375,
    "bagging_fraction": 0.8627042660815395,
    "bagging_freq": 6,
    "feature_fraction": 0.9820111656156094,
    "max_bin": 198,
    "grow_policy": "lossguide",
    "extra_trees": False,
    "drop_rate": 0.2263886508110925,
    "skip_drop": 0.3137245186015346,
    "device": "gpu",
    "verbose": -1,
}

lgb_base = Pipeline(
    [
        ("features", Features()),
        (
            "model",
            LGBMClassifier(**lgb_params),
        ),
    ]
)

models = {
    "lgb_base": lgb_base,
    "xgb_base": xgb_base,
}

In [ ]:
def pipeline_hash(pipeline):
    params = pipeline.get_params(deep=False)

    safe = {}
    for k, v in params.items():
        if isinstance(v, (int, float, str, bool, tuple)):
            safe[k] = v
        else:
            # Only keep the CLASS NAME, not the object
            safe[k] = v.__class__.__name__

    return hashlib.md5(pickle.dumps(safe)).hexdigest()


def oof_cached_fit(name, pipeline, X, y, cv, cache_dir=model_dir):
    # Create directory for this model
    new_model_dir = os.path.join(cache_dir, name)
    os.makedirs(new_model_dir, exist_ok=True)

    # Compute hash
    current_hash = pipeline_hash(pipeline)
    hash_path = os.path.join(new_model_dir, "hash.txt")
    oof_path = os.path.join(new_model_dir, "oof.npy")
    model_path = os.path.join(new_model_dir, "model.pkl")

    if os.path.exists(hash_path) and os.path.exists(oof_path) and os.path.exists(model_path):
        saved_hash = open(hash_path).read().strip()

        if saved_hash == current_hash:
            print(f"\n[LOAD] {name}: Cached model found. Skipping training.")
            oof = np.load(oof_path)
            final_model = pickle.load(open(model_path, "rb"))
            return oof, final_model

        else:
            print(f"\n[INFO] {name}: Pipeline changed → retraining.")
    else:
        print(f"\n[TRAIN] {name}: No cache → training model.")
    oof = np.zeros(len(X))

    for fold, (tr_idx, val_idx) in enumerate(cv.split(X, y)):
        print(f"{name} - Fold {fold + 1}/{cv.n_splits}")

        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        model = clone(pipeline)
        model.fit(X_tr, y_tr)
        oof[val_idx] = model.predict_proba(X_val)[:, 1]

    # Train final on full data
    final_model = clone(pipeline)
    final_model.fit(X, y)

    np.save(oof_path, oof)
    pickle.dump(final_model, open(model_path, "wb"))
    open(hash_path, "w").write(current_hash)

    print(f"[SAVED] {name}: Model + OOF cached.\n")

    return oof, final_model


def model_training(models, X, y, cv):
    oof_dict = {}
    final_dict = {}

    for name, pipe in models.items():
        oof, final_model = oof_cached_fit(name, pipe, X, y, cv)
        oof_dict[name] = oof
        final_dict[name] = final_model

    print("\nAll models trained (cached when possible).")
    return oof_dict, final_dict


best:</br>
Base model performance (OOF AUC & Accuracy):</br>
lgb_base     - ROC_AUC: 0.72751 </br>
xgb_base     - ROC_AUC: 0.72617 </br>

In [ ]:
oof_dict, final_models = model_training(models, X, y, cv)

print("\nAll base models trained successfully.\n")

# Level-1 OOF matrix
predictionMatrix = pd.DataFrame(oof_dict, index=X.index)

print("\nBase model performance (OOF AUC & Accuracy):")
for name in predictionMatrix.columns:
    auc = roc_auc_score(y, predictionMatrix[name])
    print(f"{name:12} - ROC_AUC: {auc:.5f}")

plt.figure(figsize=(10, 8))
sns.heatmap(
    data=pd.concat([predictionMatrix, y.rename(Target_Col)], axis=1).corr().abs(),
    cmap="icefire",
    annot=True,
    fmt=".2f",
)
plt.title("Correlation between base model OOF preds and target")
plt.show()

In [ ]:
for name, _ in final_models.items():
    _sub = pd.DataFrame(
        data=final_models[name].predict_proba(X_test)[:, -1],
        index=X_test.index,
        columns=[name]
    )
    _sub.to_csv(output_dir + f"submission {name}.csv")